# Project 2: Motor Effort Test to Develop an Arm Prosthesis

You just joined a computational neuroscience lab working on brain-computer interfaces.

The previous intern built a neural decoder capable of identifying 3 visual stimuli from spike train recordings. According to the intern, the model works well, with almost 100 % accuracy!

Now the lab is starting a new project. The team wants to **reuse this decoder to predict the motor phase** *(work, release or rest)* from a new dataset of neural recordings captured during a motor effort test.

The goal is to decode three motor states in real time to drive the prosthesis.

## Your course toolbox

Everything you need is in the course. When you are stuck, go back to the chapter:

| Step of this project | Where it is in the course |
|:---|:---|
| Read the raw CSV files | `data/README.md`, and `preview()` in `src/spikes.py` |
| Read rasters and PSTHs | Chapter 3: *Reading Spike Data: Raster and PSTH* |
| Spikes to counts (`sum` over an axis), counts in time windows, rate coding and what it discards | Chapter 3: *Warm-Up* (step 5: counts in time windows) and *Rate Is Not the Only Code* |
| The perceptron rule | Chapter 5 |
| Train / test split, accuracy, confusion matrix | Chapter 5: *Evaluating a Decoder* |
| Standardization: `mu`, `sigma` and why they are part of the model | Chapter 5: *Real Recordings: Standardize Your Features* |
| Weights are tied to neurons (channels) | Chapter 5: *Chapter Bilan* |
| One output neuron per class, `argmax`, the multiclass rule | Chapter 7: *More Than Two Classes*, `notebooks/v2/Chapter7_LiveExercises.ipynb` part B, step 4 (`train_multiclass`, on any features) |
| Layers, activations, and the collapse without them | Chapter 7: *What If We Remove the Step?*, part A of the notebook |

---

## 0. Setup

Run `uv sync` once at the repository root, then open this notebook with the project kernel. Everything it needs is in this folder.

You inherit three things from the previous intern:

| File | What it is |
|:---|:---|
| `src/decoder.py` | the decoder: a multiclass perceptron with `fit`, `predict`, `accuracy` |
| `models/pretrained_decoder.safetensors` | its trained parameters, in **safetensors**, the standard format to share model weights |
| `data/visual_3stim_*.csv` | the visual recordings it was trained on (grating / face / house) |

And the lab gives you the new recordings: `data/motor_effort_*.csv`. All the files are described in `data/README.md`; `src/spikes.py` loads them and provides two plotting helpers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.decoder import StimulusDecoder, load_pretrained
from src.spikes import load_dataset, plot_raster, plot_psth

vis_spikes, vis_labels, vis_names, dt = load_dataset("visual_3stim")
mot_spikes, mot_labels, mot_names, _ = load_dataset("motor_effort")

print("visual:", vis_spikes.shape, vis_names)
print("motor: ", mot_spikes.shape, mot_names)

### What is in a dataset?

The recordings are **CSV files** in `data/`, described in `data/README.md`. Open them: any spreadsheet or text editor will do.

| File | One row per | Columns |
|:---|:---|:---|
| `datasets.csv` | recording session | number of trials and neurons, trial duration, time resolution, stimulus onset |
| `<dataset>_trials.csv` | trial | `trial`, `label`, `label_name` |
| `<dataset>_spikes.csv` | spike | `trial`, `neuron`, `time_ms` |

`load_dataset(name)` turns them into arrays:

| Variable | Shape | Meaning |
|:---|:---|:---|
| `spikes` | `(n_trials, n_neurons, n_bins)` | `1` if the neuron fired in that 1 ms bin, `0` otherwise |
| `labels` | `(n_trials,)` | the class of each trial |
| `label_names` | `(n_classes,)` | human-readable names of the classes |
| `dt` | scalar | bin width in seconds |

**Have a look at the raw files** before anything else: the cell below prints the first lines of each one. Compare the two sessions.

In [ ]:
from src.spikes import preview

preview("visual_3stim")
print()
preview("motor_effort")

---

## 1. Reproduce the intern's result

In [ ]:
decoder = load_pretrained()
print(f"accuracy on the visual recordings: {decoder.accuracy(vis_spikes, vis_labels):.1%}")

---

## 2. Plug it into the motor effort test

In [ ]:
pred = decoder.predict(mot_spikes)
acc = np.mean(pred == mot_labels)
print(f"accuracy on the motor recordings: {acc:.1%}")

# Confusion matrix: rows = true motor state, columns = decoder output
confusion = np.zeros((3, 3), dtype=int)
for t, p in zip(mot_labels, pred):
    confusion[t, p] += 1
print("rows:", mot_names, "| columns:", decoder.label_names)
print(confusion)

The prosthesis would be driven by this output. Clearly, something is wrong.

---

## 3. Audit the decoder

**Your turn.** Read `src/decoder.py` line by line, as if you had to sign off on it before it drives a real arm. Find every reason why it cannot work on the motor recordings, and for each one, explain *why* it matters, **computationally and biologically**.

Some leads:
- Compare the two datasets: shapes, trial duration, firing rates. What does the decoder assume about its input?
- Read the model card of `models/pretrained_decoder.safetensors` (`print_model_card()` below): what does the file declare about itself?
- What do `mu` and `sigma` contain, and where do they come from? *(Chapter 5: they are part of the model.)*
- The weight `W[c, i]` multiplies channel `i`. What does channel `i` record in each dataset? *(Chapter 5 bilan: weights describe neurons.)*
- What does the output of `predict` mean in the motor experiment? *(Chapter 7: one output neuron per class.)*
- How was the "100 % accuracy" measured? *(Chapter 5: train vs test.)*

*Your audit here.*

In [ ]:
from src.decoder import print_model_card

print_model_card()

# Explore here: compare datasets, inspect decoder.mu / decoder.sigma / decoder.W, ...

---

## 4. Retrain on the motor data

`StimulusDecoder` has a `fit` method: read it with Chapter 7's multiclass rule in mind. Retrain a fresh decoder on the motor recordings, with a **proper train / test split**, and look at the confusion matrix again.

In [ ]:
# TODO: split the motor trials into train / test sets
# TODO: fit a new StimulusDecoder(label_names=mot_names) on the training set
# TODO: print the test accuracy and the confusion matrix

Is the retrained decoder good enough to control a prosthesis? Which motor states does it still confuse?

---

## 5. Look at the neurons

Before changing the model, look at what the neurons actually do in each motor state. Browse through the channels with rasters and PSTHs.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_raster(mot_spikes, mot_labels, neuron=0, label_names=mot_names, dt=dt, ax=axes[0])
plot_psth(mot_spikes, mot_labels, neuron=0, label_names=mot_names, dt=dt, ax=axes[1])
plt.tight_layout()

**Your turn.**

1. Find the neurons that distinguish the motor states. How do they do it?
2. Explain why the decoder's features cannot separate some of the states, **whatever the weights**.
3. Design better features, still feeding a (multiclass) perceptron, and show they solve the problem on held-out trials. *(Chapter 3: what does rate coding discard? Warm-up step 5 counts spikes in time windows, and `train_multiclass`, Chapter 7 step 4, learns on any features.)*
4. Discuss: what does a "real-time" decoder need that this offline one does not have? *(Chapter 3, Rate Is Not the Only Code: a count needs a window.)*

In [ ]:
# Your investigation starts here.

---

## 6. Your bridge

The course used one tool all week: the epistemological bridge. Fill it **for your pipeline**, the inherited decoder **and** the one you fixed: for each piece of biological reality, what did each of them turn it into, and was it kept, simplified or discarded?

| Biological reality | Inherited decoder | Your fixed decoder | Kept, simplified or discarded? |
|:---|:---|:---|:---|
| Spike trains over time | ... | ... | ... |
| Synaptic strength and sign | ... | ... | ... |
| Each neuron's identity (channel) | ... | ... | ... |
| Trial-to-trial variability (noise) | ... | ... | ... |
| How the brain learns | ... | ... | ... |
| Competition between motor states | ... | ... | ... |
| *(add your own rows)* | ... | ... | ... |

---

## Bonus: does it matter where you put activations if you mix them across layers?

The lab considers adding a **hidden layer** to the decoder. You can't train a hidden layer yet (that needs backpropagation, Chapter 7), but you can **fix** it with random weights and train only the output layer with the multiclass perceptron rule (Chapter 6 bilan, step 4: learn the last layer on the hidden representation; Chapter 7 step 4: `train_multiclass`). Use `forward` / `layer` from `notebooks/v2/Chapter7_LiveExercises.ipynb`.

Compare, on held-out trials:
1. a hidden layer **without** activation, then the output layer;
2. a hidden layer **with a ReLU**, then the output layer;
3. **no** hidden layer, but an activation (sigmoid, ReLU) applied to the **output scores** before the `argmax`.

Do it on the spike counts, then on your better features. Which placements change the predictions, which don't, and why? *(Chapter 7: the collapse without non-linearity; and think about what `argmax` does with a monotonic function.)* Can **any** activation, anywhere, separate release from rest using spike counts only? What does that tell you about features versus architecture?

In [ ]:
rng = np.random.default_rng(0)

# A fixed random hidden layer: 200 neurons reading the n_features inputs
# n_features = ...
# W_h, b_h = rng.normal(size=(200, n_features)) / np.sqrt(n_features), rng.normal(size=200)

# Bonus: your experiments here

<div style="border-left: 4px solid #f59e0b; padding: 0.5em 1em; background: rgba(245,158,11,0.08);">
<b>Deliverables, 20 November (article + notebook) and 23 November (presentation)</b>: a <b>mini-article</b> (4 pages, see <code>REPORT_TEMPLATE.md</code>) with this notebook, cleaned, in annex, and its <b>30-minute presentation</b> at the lab meeting. Both show (1) your audit of the inherited decoder, every flaw with the evidence that shows it, (2) your fixed decoder and its <b>held-out</b> performance, (3) what this says about how motor information is encoded compared to the visual stimuli, and (4) what each function of the pipeline changes, biologically and computationally.
</div>